# 12. Revisión asistida de consultas de recuperación
Esta libreta documenta la primera etapa de la evaluación: verificar los datos, conservar las consultas que están bien planteadas y precisar sus criterios de pertinencia.

**Resultado:** 24 formulaciones para 12 necesidades. Se conservan 22 textos y se aclara una conjunción en las dos versiones de N02. Las decisiones ya están incorporadas; no hay celdas que exijan revisar manualmente las 24 preguntas.

## 1. Resultado de la revisión de los datos

| Comprobación | Resultado |
|---|---:|
| Filas del canónico: artículo–autor | 1,151 |
| Artículos del catálogo | 519 |
| Identificadores de artículo distintos | 519 |
| Nombres de autor distintos por coincidencia exacta | 628 |
| Pares artículo–autor distintos | 1,151 |
| Comparaciones bibliográficas artículo–campo | 3,633 |
| Diferencias en esas comparaciones | 0 |
| Diferencias en los conjuntos de autores por artículo | 0 |
| Artículos con texto en `abstract` | 519 |
| Textos de `abstract` distintos | 514 |
| DOI vacíos en el catálogo | 25 |
| Keywords vacías en el catálogo | 5 |

Se compararon título, año, área, keywords, abstract, DOI y URL de cada artículo contra todas sus filas del canónico.

Los archivos recibidos coinciden exactamente con las huellas de la auditoría:

## 2. Decisiones sobre las 12 necesidades

Los números de artículo de la tabla son **referencias de lectura y contraste**.
La tabla no es una lista exhaustiva de relevantes ni asigna etiquetas finales.

| Necesidad | Decisión | Motivo y contraste |
|---|---|---|
| N01 | Conservar | 117 aporta recuperación sobre documentación; 175 permite contrastar otro dominio de preguntas y respuestas. Usar un manual, como en 140, no demuestra recuperación. |
| N02 | Aclarar ES/EN | Recursos o métodos. 314 aporta el recurso; 129, clasificación. En 129 el español está explícito en el título, no en el abstract. |
| N03 | Conservar | 355 vincula recorte y cotas de privacidad. 353 y 515 comparten conceptos, sin demostrar por ello la configuración solicitada. |
| N04 | Conservar | 324 transforma textos en grafos. 262 permite estudiar una aplicación textual distinta; 214 clasifica temas sin demostrar esa transformación. |
| N05 | Conservar | 285 respalda síntesis desde requisitos en lenguaje natural. 330 no explicita esa entrada; 312 transforma modelos formales. |
| N06 | Conservar | El resumen completo de 316 confirma validación de integración de fuentes. El fragmento Context. de la auditoría no era el resumen completo. |
| N07 | Conservar | 336 reúne plataforma, oscilaciones y coherencia. 444 aporta un método concreto; 446 no acredita la combinación de análisis requerida. |
| N08 | Conservar | El artículo 1 sí declara forecasting y su punto de fallo. 67 y 99 no tratan la misma combinación de problema físico y método. |
| N09 | Conservar | 4 conecta ESG, riesgo y rendimiento. 487 no acredita ESG; sostenibilidad en 260 no equivale a decisiones de portafolio. |
| N10 | Conservar | 271 conecta topología y aprendizaje contrastivo. 386 trata hipergrafos cerebrales, lo que no acredita esa conexión. |
| N11 | Conservar | 337 sustenta ocultamiento, clave y paralelismo. 350 permite contrastar un aporte parcial. In parallel en 342 no demuestra paralelización computacional. |
| N12 | Conservar | 482 aborda el número de policías en grafos de fichas de caminos y estrellas. 501 trata guardias geométricos; 339, tokens financieros. |

## 3. Reglas fijadas para este piloto

- **Campo de recuperación:** solamente `abstract` para BM25 y los 11 modelos densos, igual que en los índices existentes.
- **Información para revisar pertinencia:** título y abstract del catálogo, disponibles por igual para todos los sistemas. El título puede aclarar un atributo omitido en el resumen, como el idioma en 129. Esto no añade títulos a la representación documental ni exige recalcular vectores.
- **Grados previstos:** 2 = aporte directo; 1 = aporte parcial concreto; 0 = sin aporte pertinente. Información insuficiente o un caso no revisado conservan estado pendiente: no son ceros automáticos. No se asignan aquí grados finales a artículos.
- **División:** N01–N04 desarrollo; N05–N12 comparación. ES y EN de una misma necesidad permanecen en el mismo grupo. N01 ya se utilizó en pilotos.
- **Unidad de análisis:** 12 necesidades bilingües, no 24 necesidades independientes. El grupo de comparación contiene ocho necesidades.
- **Procedencia:** las preguntas iniciales fueron propuestas después de conocer el corpus. La revisión actual también usa ese corpus. Se trata de un conjunto interno y acotado de recuperación temática, no de consultas externas inéditas.
- **Consulta al recuperador:** enviar únicamente el campo `consulta`; después, cada modelo añadirá su formato técnico. No enviar criterios ni ejemplos.
- **Alcance de la selección:** todavía no se evalúa recuperación para pares de autores ni la generación de justificaciones con Gemma. Un documento temático no demuestra qué variables produjeron la predicción de coautoría.

In [1]:
import hashlib
import json
from pathlib import Path
import pandas as pd

# La libreta funciona desde 07_RAG o desde la raíz del repositorio.
REPO = next((p for p in (Path.cwd(), Path.cwd().parent)
             if (p / "06_grafos/catalogo_articulos.csv").is_file()
             and (p / "07_RAG/auditoria_catalogo.json").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Abre la libreta desde 07_RAG o desde Tesis_Multimodelo.")

BASE = REPO / "07_RAG"
RUTA_CATALOGO = REPO / "06_grafos/catalogo_articulos.csv"
RUTA_AUDITORIA = BASE / "auditoria_catalogo.json"
RUTA_CONSULTAS = BASE / "consultas_para_revision.csv"
RUTA_SALIDA = BASE / "consultas_evaluacion_v1.csv"

def sha256(ruta):
    return hashlib.sha256(ruta.read_bytes()).hexdigest()

# Los identificadores se leen como texto. Los vacíos permanecen vacíos.
catalogo = pd.read_csv(RUTA_CATALOGO, dtype=str, keep_default_na=False)
consultas = pd.read_csv(RUTA_CONSULTAS, dtype=str, keep_default_na=False)
auditoria = json.loads(RUTA_AUDITORIA.read_text(encoding="utf-8-sig"))
sha_entradas = {p: sha256(p) for p in (RUTA_CATALOGO, RUTA_AUDITORIA, RUTA_CONSULTAS)}

SHA_CATALOGO = "74040d185b762721c29521fdc3a270740569f009b7ae40dbce654366b576598b"
SHA_CONSULTAS_ORIGINALES = "290db30d0549bcc4a935d53e5460aa33ff0075eb8177f8a9fb749cf723cdc859"
assert sha_entradas[RUTA_CATALOGO] == SHA_CATALOGO, "El catálogo no es la versión revisada."
assert SHA_CATALOGO == auditoria["fuentes"]["catalogo"]["sha256_bytes"]
assert sha_entradas[RUTA_CONSULTAS] == SHA_CONSULTAS_ORIGINALES, "El borrador original cambió."
assert len(catalogo) == 519 and catalogo["article_id"].nunique() == 519
assert catalogo["abstract"].str.strip().ne("").all()
assert catalogo["abstract"].nunique() == 514
assert len(consultas) == 24 and consultas["query_id"].nunique() == 24
assert set(consultas["necesidad_id"]) == {f"N{i:02d}" for i in range(1, 13)}
for necesidad, grupo in consultas.groupby("necesidad_id"):
    assert len(grupo) == 2 and set(grupo["idioma"]) == {"es", "en"}
    assert set(grupo["query_id"]) == {necesidad + "-ES", necesidad + "-EN"}
    division = "desarrollo" if necesidad in {"N01", "N02", "N03", "N04"} else "comparacion"
    assert set(grupo["division_propuesta"]) == {division}

print("Entradas verificadas: 519 artículos, 514 textos y 12 necesidades ES/EN.")
print("Todos los archivos se han abierto solo para lectura.")

Entradas verificadas: 519 artículos, 514 textos y 12 necesidades ES/EN.
Todos los archivos se han abierto solo para lectura.


In [2]:
# Si el canónico está en su ubicación registrada, repetimos la correspondencia.
# Su ausencia local no impide usar el catálogo cuya huella ya fue verificada.
RUTA_CANONICO = REPO / auditoria["fuentes"]["canonico"]["ruta_relativa"]
if RUTA_CANONICO.is_file():
    sha_entradas[RUTA_CANONICO] = sha256(RUTA_CANONICO)
    assert sha_entradas[RUTA_CANONICO] == auditoria["fuentes"]["canonico"]["sha256_bytes"]
    canonico = pd.read_csv(RUTA_CANONICO, dtype=str, keep_default_na=False)
    assert len(canonico) == 1151
    assert set(canonico["indice"]) == set(catalogo["article_id"])
    assert not canonico.duplicated(["indice", "Autor_norm"]).any()
    equivalencias = {"titulo": "Titulo", "anio": "Año", "area": "Area",
                     "keywords": "Keywords", "abstract": "Abstract", "doi": "Doi", "url": "URL"}
    diferencias = []
    for articulo in catalogo.to_dict("records"):
        filas = canonico[canonico["indice"] == articulo["article_id"]]
        for campo_catalogo, campo_canonico in equivalencias.items():
            if set(filas[campo_canonico]) != {articulo[campo_catalogo]}:
                diferencias.append((articulo["article_id"], campo_catalogo))
        autores = {nombre.strip() for nombre in articulo["autores"].split(";") if nombre.strip()}
        if autores != set(filas["Autor_norm"]):
            diferencias.append((articulo["article_id"], "autores"))
    assert not diferencias, f"Diferencias entre catálogo y canónico: {diferencias[:10]}"
    print("Canónico verificado: 3,633 comparaciones bibliográficas y 519 conjuntos de autores sin diferencias.")
else:
    print("Canónico no encontrado en su ruta registrada; se omite únicamente esa comprobación local.")

# Mostrar coincidencias no modifica ni fusiona ningún artículo.
repetidos = catalogo[catalogo.duplicated("abstract", keep=False)]
display(repetidos[["article_id", "titulo", "anio", "doi"]])

Canónico verificado: 3,633 comparaciones bibliográficas y 519 conjuntos de autores sin diferencias.


,article_id,titulo,anio,doi
195,196,Inference of the Mass Composition of Cosmic Ra...,2024,10.48550/arxiv.2406.06315
196,197,Inference of the Mass Composition of Cosmic Ra...,2025,10.1103/physrevlett.134.021001
199,200,La Leaderboard: A Large Language Model Leaderb...,2025,10.18653/v1/2025.acl-long.1561
200,201,La Leaderboard: A Large Language Model Leaderb...,2025,10.48550/arxiv.2507.00999
303,304,Expanding the Pierre Auger Observatory Open Da...,2025,10.22323/1.501.1251
304,305,Expanding the Pierre Auger Observatory Open Da...,2025,10.48550/arxiv.2507.08504
388,389,Cloud Detection using Night Sky Background Lig...,2025,10.22323/1.501.0408
389,390,Cloud Detection using Night Sky Background Lig...,2025,10.48550/arxiv.2509.06343
473,474,Asynchronous Fault-Tolerant Language Decidabil...,2025,10.1145/3732772.3733514
474,475,Asynchronous Fault-Tolerant Language Decidabil...,2025,10.48550/arxiv.2502.00191


## 4. Criterios de pertinencia revisados

El siguiente diccionario contiene una rúbrica por necesidad. Se comparte entre español e inglés. Las reglas distinguen aportes directos, parciales y ajenos a la necesidad; no exigen que un único artículo responda por completo toda la pregunta.

In [3]:
# Criterios documentados; no son etiquetas de los artículos.
REVISION = {
    "N01": {
        "criterio_grado_2": "Presenta, implementa o estudia consulta en lenguaje natural sobre documentación técnica con respuestas basadas en material recuperado; el resumen o título+resumen permite reconocer los tres requisitos centrales.",
        "criterio_grado_1": "Aporta un método directamente útil de preguntas/respuestas con contexto recuperado o de uso de evidencia textual para responder, pero falta la aplicación a documentación técnica o solo aborda una parte sustantiva del proceso. No basta compartir la palabra chatbot.",
        "criterio_grado_0": "Chatbots, UX, corpus, documentación o recuperación en otros sentidos sin contribución identificable a responder preguntas usando evidencia documental recuperada.",
        "observaciones_revision": "Conservar. Documentación técnica, preguntas en lenguaje natural y fundamentación con material recuperado. No exigir Debian ni un modelo concreto. N01 ya se utilizó en pilotos y permanece en desarrollo."
    },
    "N02": {
        "criterio_grado_2": "Ofrece un recurso de informes mamográficos en español anotados/organizados con BI-RADS, o un método de PLN aplicado a su organización/clasificación con ese esquema. Un dataset puede ser directo sin proponer un clasificador.",
        "criterio_grado_1": "Aporta una parte verificable del problema de PLN sobre informes mamográficos o BI-RADS, por ejemplo un método pertinente cuyo idioma distinto del español esté explícito. Explicar el aporte y el requisito que no cubre. El PLN clínico general no cuenta como aporte parcial por sí solo. Si el idioma no consta, mantener esa comprobación pendiente en vez de asignar un grado por la omisión.",
        "criterio_grado_0": "Procesamiento exclusivo de imágenes mamográficas, detección de lesiones, radiómica o clasificación clínica sin trabajar el contenido textual solicitado; recursos de otros dominios clínicos sin vínculo metodológico concreto con la necesidad.",
        "observaciones_revision": "Ajuste en ambos idiomas: recursos o métodos. Un recurso y un clasificador pueden atender la necesidad por separado. En el artículo 129, español se confirma en el título y no se explicita en el abstract. Aplicar juicio con título y abstract de manera uniforme. Si el idioma no está acreditado en ninguno de esos campos, dejar pendiente esa comprobación; no confundirla con un aporte parcial ya verificado."
    },
    "N03": {
        "criterio_grado_2": "Analiza garantías o pérdida de privacidad del entrenamiento por descenso de gradiente con recorte; explica o presenta cotas, contabilidad u otro análisis formal de esa configuración.",
        "criterio_grado_1": "Estudia formalmente la privacidad del entrenamiento por gradientes o el papel del recorte y proporciona una contribución relevante a la pregunta, aunque no cubra todos sus requisitos. Una mención general a privacidad diferencial o divergencias no basta.",
        "criterio_grado_0": "Optimización con gradientes sin garantías de privacidad, privacidad aplicada sin entrenamiento por gradiente, seguridad/cifrado, opiniones sobre privacidad o usos léxicos de privado.",
        "observaciones_revision": "Conservar. Se requieren garantías de privacidad vinculadas al entrenamiento por gradientes y al recorte. Compartir privacidad diferencial o una divergencia matemática no basta."
    },
    "N04": {
        "criterio_grado_2": "Presenta una herramienta/procedimiento de texto a grafos y describe su uso o capacidad para recuperación de información o clasificación temática. No hace falta implementar ambas tareas.",
        "criterio_grado_1": "Construye grafos a partir de documentos y aporta detalles metodológicos transferibles, pero la aplicación descrita es otra tarea textual, como autoría o contenido generado. Explicar la diferencia de objetivo.",
        "criterio_grado_0": "Construye grafos de datos no textuales, o describe explícitamente representación de texto o clasificación temática mediante un procedimiento ajeno a grafos. La falta de detalle sobre cómo se construye un grafo se mantiene pendiente, sin convertirla automáticamente en grado 0.",
        "observaciones_revision": "Conservar. Debe existir conversión de texto a grafos. Recuperación de información o clasificación temática son alternativas. No exigir text2graphAPI ni confundir cualquier GNN con conversión de documentos. Si título y abstract no permiten comprobar la conversión, mantener pendiente: la omisión no implica grado 0."
    },
    "N05": {
        "criterio_grado_2": "Trata explícitamente generación/síntesis automática de programas o código desde requisitos en lenguaje natural y examina sus dificultades o limitaciones. Una revisión y un estudio experimental pueden cumplirlo por igual.",
        "criterio_grado_1": "Aporta evidencia útil a un componente: por ejemplo, generación automática de código y sus dificultades sin confirmar que la entrada sean requisitos en lenguaje natural; o programación por reglas en lenguaje cotidiano con verificación, sin generación explícita de programas completos.",
        "criterio_grado_0": "NLP/LLM sin generación de programas; medición, gestión, clasificación o traducción formal de modelos que no aborde la generación solicitada desde lenguaje natural.",
        "observaciones_revision": "Conservar. La entrada en lenguaje natural debe quedar sustentada; usar un LLM para generar código no la acredita por sí solo. Admitir tanto revisiones como métodos."
    },
    "N06": {
        "criterio_grado_2": "Aborda explícitamente la validación o adecuación de integrar bases/fuentes distintas para construir o mejorar modelos de estimación de esfuerzo/costo de software.",
        "criterio_grado_1": "Aporta una pieza operativa para esa misma tarea, como comparar datos de proyectos/fuentes para estimación de esfuerzo, estudiar heterogeneidad o evaluar integración en ese contexto, pero sin resolver explícitamente la decisión de adecuación.",
        "criterio_grado_0": "Fusión o adaptación entre dominios ajenos a estimación de software; métodos genéricos de ML o medición funcional de software sin vínculo explícito con combinar/validar fuentes para estimar esfuerzo.",
        "observaciones_revision": "Conservar. El resumen completo de 316 sí describe validación de integración de fuentes para estimación de esfuerzo/costo. No introducir Kruskal–Wallis en la consulta. Sus cifras porcentuales contienen una inconsistencia aritmética que debe verificarse antes de citarlas."
    },
    "N07": {
        "criterio_grado_2": "Describe una plataforma/herramienta integrada que procese mediciones de sistemas eléctricos y soporte explícitamente análisis de oscilaciones y agrupación por coherencia dinámica.",
        "criterio_grado_1": "Aporta un método o componente concreto para la plataforma solicitada: identificación modal desde señales eléctricas, clustering de unidades a partir de modos/oscilaciones, o análisis regional de frecuencia; no confirma la plataforma integrada o todos sus análisis.",
        "criterio_grado_0": "Comparte redes eléctricas, grafos, simulación o calidad de energía pero no aporta evidencia sobre los análisis dinámicos solicitados. Agrupar exclusivamente por conectividad topológica no establece coherencia dinámica.",
        "observaciones_revision": "Conservar. Se busca una plataforma para sistemas eléctricos con análisis de oscilaciones y coherencia dinámica. Los métodos que resuelven un componente pueden aportar evidencia parcial. Una comunidad topológica no demuestra coherencia dinámica."
    },
    "N08": {
        "criterio_grado_2": "Estudia modelos neuronales de ecuaciones diferenciales para la estructura de enanas blancas y analiza explícitamente los límites o fallos del pronóstico o la extrapolación.",
        "criterio_grado_1": "Aborda el mismo problema físico mediante modelos neuronales de ecuaciones diferenciales, pero el abstract no evalúa los límites del pronóstico; o aporta una parte explícita y utilizable del análisis solicitado.",
        "criterio_grado_0": "Redes neuronales para otras ecuaciones/fenómenos; astronomía con otro objetivo; ODE o análisis de fallo sin modelo neuronal ni relación con enanas blancas.",
        "observaciones_revision": "Conservar pronóstico/forecasting: el resumen completo del artículo 1 usa ambos términos y describe un forecasting breakdown point. No inferir que se predice la evolución temporal de una estrella; el resumen describe densidad frente a distancia al centro."
    },
    "N09": {
        "criterio_grado_2": "Describe o analiza cómo se incorpora ESG a la construcción o selección de un portafolio considerando riesgo y rendimiento. El documento puede utilizar distintos métodos; no se exige replicar el procedimiento del orientador.",
        "criterio_grado_1": "Aporta una parte explícita y utilizable del problema financiero, por ejemplo una evaluación ESG vinculada a la decisión de portafolio que no desarrolla el compromiso riesgo-rendimiento, o una optimización de ese compromiso que no incorpora ESG. No basta un método genérico de optimización sin aplicación financiera.",
        "criterio_grado_0": "Trata sostenibilidad, gobernanza, riesgo, inversión o optimización en otros problemas, sin un aporte concreto a la necesidad de portafolio definida.",
        "observaciones_revision": "Conservar. ESG debe vincularse con decisiones de portafolio y el compromiso entre riesgo y rendimiento. Sostenibilidad o gobernanza en otros dominios no bastan."
    },
    "N10": {
        "criterio_grado_2": "Describe un método que integra información topológica de estructuras de orden superior en aprendizaje contrastivo de grafos, o analiza específicamente esa integración.",
        "criterio_grado_1": "Desarrolla un componente metodológico concreto y utilizable de la integración: por ejemplo aprendizaje contrastivo de grafos con una representación topológica no demostrada de orden superior, o una representación topológica de grafos explícitamente destinada a un esquema de aprendizaje contrastivo aún no desarrollado. Debe existir el vínculo técnico con la necesidad, no solo una palabra compartida.",
        "criterio_grado_0": "Usa redes neuronales de grafos, hipergrafos, complejos simpliciales, interacciones de orden superior o TDA en otros problemas, sin aportar una conexión concreta con aprendizaje contrastivo de grafos.",
        "observaciones_revision": "Conservar. Debe existir una conexión técnica entre información topológica de orden superior y aprendizaje contrastivo de grafos. Hipergrafos o GNN en otro problema no bastan. No exigir TopoGCL."
    },
    "N11": {
        "criterio_grado_2": "El texto disponible sustenta conjuntamente ocultamiento en imágenes, clave secreta y paralelización de la codificación/inserción.",
        "criterio_grado_1": "Describe un método concreto de ocultamiento o inserción de datos en imágenes que aporta parte de la necesidad, pero no acredita la clave secreta, la paralelización o ambas. Una mejora de tiempo por sí sola no acredita paralelismo. La marca de agua puede ser parcial si efectivamente inserta datos; no se decide por su etiqueta.",
        "criterio_grado_0": "Solo cifra imágenes o archivos, gestiona claves, autentica mediante características externas sin inserción acreditada, o usa procesamiento paralelo sin ocultar datos en una imagen. La seguridad general no es suficiente.",
        "observaciones_revision": "Conservar. Distinguir ocultamiento de datos, clave secreta y paralelización de la codificación. In parallel en el artículo 342 no acredita procesamiento paralelo computacional. Cifrado y zero-watermarking no equivalen automáticamente a ocultamiento por inserción."
    },
    "N12": {
        "criterio_grado_2": "Analiza, caracteriza, acota o determina el cop number de grafos de fichas cuyos grafos base son caminos o estrellas, o desarrolla una estrategia de captura para esas familias.",
        "criterio_grado_1": "Proporciona una herramienta, equivalencia o cota para el juego en grafos de fichas y establece explícitamente su aplicación al caso de caminos o estrellas, pero no resuelve ni desarrolla el resultado solicitado. No basta que trate otra familia de grafos.",
        "criterio_grado_0": "Trata grafos o caminos sin el juego correspondiente, otros juegos de vigilancia geométrica, tokens financieros, o grafos de fichas de otras familias sin relación explícita con caminos o estrellas.",
        "observaciones_revision": "Conservar. Grafos de fichas significa token graphs construidos a partir de caminos o estrellas. O permite cualquiera de las dos familias. Cop number es el mínimo de policías. No exigir también estrellas subdivididas ni ampliar a otros problemas de vigilancia."
    }
}

# Únicos cambios del texto original.
AJUSTES = {
    "N02-ES": "Busco recursos o métodos de procesamiento del lenguaje para organizar o clasificar informes de mamografía en español según BI-RADS.",
    "N02-EN": "I am looking for language-processing resources or methods to organize or classify Spanish mammography reports according to BI-RADS."
}

# Referencias para inspeccionar ejemplos y fronteras; no son listas de relevantes.
LECTURAS = {
    "N01": [
        "117",
        "175",
        "140"
    ],
    "N02": [
        "129",
        "314",
        "161"
    ],
    "N03": [
        "355",
        "353",
        "515"
    ],
    "N04": [
        "324",
        "262",
        "214"
    ],
    "N05": [
        "285",
        "330",
        "312"
    ],
    "N06": [
        "316",
        "309",
        "213"
    ],
    "N07": [
        "336",
        "444",
        "446"
    ],
    "N08": [
        "1",
        "67",
        "99"
    ],
    "N09": [
        "4",
        "487",
        "260"
    ],
    "N10": [
        "271",
        "386",
        "262"
    ],
    "N11": [
        "337",
        "350",
        "342",
        "362"
    ],
    "N12": [
        "482",
        "501",
        "339"
    ]
}

In [4]:
# Se crea una tabla nueva. El borrador original no se edita.
evaluacion = consultas[["query_id", "necesidad_id", "division_propuesta", "idioma", "consulta_propuesta"]].copy()
evaluacion = evaluacion.rename(columns={"division_propuesta": "division", "consulta_propuesta": "consulta_original"})
evaluacion["consulta"] = [AJUSTES.get(qid, texto)
                          for qid, texto in zip(evaluacion["query_id"], evaluacion["consulta_original"])]
for columna in ("criterio_grado_2", "criterio_grado_1", "criterio_grado_0", "observaciones_revision"):
    evaluacion[columna] = evaluacion["necesidad_id"].map(lambda n: REVISION[n][columna])

evaluacion["estado_revision"] = "revisada_por_asistente"
evaluacion["validacion_humana"] = "no_realizada"
evaluacion["campos_para_juicio"] = "titulo+abstract"
evaluacion["version"] = "CONSULTAS_RAG_V1_REVISION_ASISTIDA"
evaluacion["sha256_catalogo"] = SHA_CATALOGO

# Verificar exactamente qué cambió y mantener las parejas lingüísticas juntas.
cambios = evaluacion[evaluacion["consulta"] != evaluacion["consulta_original"]]
assert set(cambios["query_id"]) == {"N02-ES", "N02-EN"}
assert evaluacion["consulta"].str.strip().ne("").all()
assert evaluacion.groupby("necesidad_id")["division"].nunique().eq(1).all()
print("Consultas preparadas:", len(evaluacion), "| textos conservados:", len(evaluacion) - len(cambios))
with pd.option_context("display.max_colwidth", None):
    display(cambios[["query_id", "consulta_original", "consulta"]])
display(evaluacion.groupby("division").agg(necesidades=("necesidad_id", "nunique"),
                                           formulaciones=("query_id", "count")))

Consultas preparadas: 24 | textos conservados: 22


,query_id,consulta_original,consulta
2,N02-ES,Busco recursos y métodos de procesamiento del lenguaje para organizar o clasificar informes de mamografía en español según BI-RADS.,Busco recursos o métodos de procesamiento del lenguaje para organizar o clasificar informes de mamografía en español según BI-RADS.
3,N02-EN,I am looking for language-processing resources and methods to organize or classify Spanish mammography reports according to BI-RADS.,I am looking for language-processing resources or methods to organize or classify Spanish mammography reports according to BI-RADS.


,necesidades,formulaciones
division,,
comparacion,8,16
desarrollo,4,8


In [5]:
def mostrar_necesidad(necesidad_id):
    """Muestra textos, criterios y ejemplos completos, sin llamar a ningún modelo."""
    filas = evaluacion[evaluacion["necesidad_id"] == necesidad_id]
    if filas.empty:
        raise ValueError("Usa un identificador entre N01 y N12.")
    for fila in filas.to_dict("records"):
        print(f"{fila['query_id']}: {fila['consulta']}\n")
    for grado in (2, 1, 0):
        print(f"Criterio {grado}: {REVISION[necesidad_id]['criterio_grado_' + str(grado)]}\n")
    print("Observación:", REVISION[necesidad_id]["observaciones_revision"])
    print("\nReferencias de lectura; no son juicios finales ni respuestas exhaustivas:")
    for article_id in LECTURAS[necesidad_id]:
        articulo = catalogo.set_index("article_id").loc[article_id]
        print(f"\nArtículo {article_id}: {articulo['titulo']}\n{articulo['abstract']}\n")

# Vista opcional. Cambia N02 por otra necesidad si deseas consultar sus detalles.
mostrar_necesidad("N02")

N02-ES: Busco recursos o métodos de procesamiento del lenguaje para organizar o clasificar informes de mamografía en español según BI-RADS.

N02-EN: I am looking for language-processing resources or methods to organize or classify Spanish mammography reports according to BI-RADS.

Criterio 2: Ofrece un recurso de informes mamográficos en español anotados/organizados con BI-RADS, o un método de PLN aplicado a su organización/clasificación con ese esquema. Un dataset puede ser directo sin proponer un clasificador.

Criterio 1: Aporta una parte verificable del problema de PLN sobre informes mamográficos o BI-RADS, por ejemplo un método pertinente cuyo idioma distinto del español esté explícito. Explicar el aporte y el requisito que no cubre. El PLN clínico general no cuenta como aporte parcial por sí solo. Si el idioma no consta, mantener esa comprobación pendiente en vez de asignar un grado por la omisión.

Criterio 0: Procesamiento exclusivo de imágenes mamográficas, detección de lesion

## 5. Guardar la versión de trabajo

In [6]:
# Un único archivo de salida; nunca escribir sobre las entradas.
assert RUTA_SALIDA not in sha_entradas
contenido = evaluacion.to_csv(index=False, lineterminator="\n").encode("utf-8-sig")
if RUTA_SALIDA.exists():
    if RUTA_SALIDA.read_bytes() != contenido:
        raise FileExistsError("Ya existe una versión distinta. Consérvala y usa un nombre v2 para nuevos cambios.")
    print("La versión guardada ya coincide; se conserva el archivo existente.")
else:
    RUTA_SALIDA.write_bytes(contenido)
    print("CSV creado:", RUTA_SALIDA.name)

assert all(sha256(ruta) == valor for ruta, valor in sha_entradas.items()), "Una entrada cambió durante la ejecución."
print("SHA-256 consultas:", sha256(RUTA_SALIDA))
print("24 consultas / 12 necesidades / 22 textos originales conservados / 2 ajustes en N02.")
print("Estado: revisión asistida completada. Validación humana independiente: no realizada.")
print("Entradas conservadas. Siguiente actividad: BM25 sobre los mismos abstracts.")

CSV creado: consultas_evaluacion_v1.csv
SHA-256 consultas: 1e35b789207b49ebb3e1e088c342cda234c2153f7d1b50f0d05adbf8cf459f17
24 consultas / 12 necesidades / 22 textos originales conservados / 2 ajustes en N02.
Estado: revisión asistida completada. Validación humana independiente: no realizada.
Entradas conservadas. Siguiente actividad: BM25 sobre los mismos abstracts.


## Siguiente actividad

La siguiente libreta preparará BM25 sobre los mismos resúmenes y utilizará el campo `consulta` de esta versión. La codificación de consultas de los 11 modelos, RRF y la evaluación de los resultados corresponden a etapas posteriores.

Los criterios definidos aquí facilitarán esa evaluación, pero las métricas de recuperación necesitan juicios por documento. Una revisión exclusivamente asistida deberá reportarse con esa procedencia y con sus límites. No se calcularán Recall, Precision, MRR o nDCG a partir de los artículos orientadores como si fueran todos los relevantes.

## Anexo. Evidencia de la revisión documental

Los siguientes casos documentan por qué se fijaron los criterios. Son interpretaciones asistidas de los textos disponibles, no juicios finales ni una lista exhaustiva de pertinentes. Los fragmentos se contrastaron literalmente con el catálogo recibido.


### N01


**Artículo 117: A retrieval-augmented-generation pipeline to help users query system-provided documentation**


Describe exactamente recuperación sobre documentación y suministro de evidencia al generador; las páginas de manual son una implementación particular.


Abstract:

> Manual pages are segmented and embedded for dense retrieval, and the retrieved material is supplied to a language model for answer generation


**Artículo 175: Exploring the Behavior and Performance of Large Language Models: Can LLMs Infer Answers to Questions Involving Restricted Information?**


Mecanismo de QA apoyado en un pasaje recuperado; la fuente es la Biblia y no documentación técnica.


Abstract:

> The RAG system matches a question with the most similar passage from The Bible and feeds the pair to each LLM.


**Artículo 184: GIL-IIMAS UNAM at SemEval-2025 Task 3: MeSSI: A Multimodule System to Detect Hallucinated Segments in Trivia-like Inquiries.**


Estudia comprobación de respaldo y alucinaciones. Es apoyo metodológico parcial si el protocolo admite esta parte del flujo; no resuelve consulta de manuales.


Abstract:

> retrieves supporting Wikipedia material, creates a reference context and compares linguistic information in the question against that evidence


**Artículo 273: UCOM_UNAM_PLN at CheckThat! 2025: Evaluating LLMs in a two-Step Architecture for Numerical Fact Checking**


La recuperación apoya verificación numérica, no preguntas sobre documentación. El vínculo es el uso de evidencia recuperada; definir antes de anotar si alcanza el umbral parcial.


Abstract:

> evidence retrieval through BM25-preselected documents


**Artículo 140: Chatbot Mar for Psychological First Aid: An Innovation in Immediate Crisis Care following Traffic Accidents**


Usar un manual de intervención no demuestra recuperación en tiempo de consulta, y el dominio no es documentación técnica de sistemas.


Abstract:

> Based on the ABCDE Crisis Intervention Manual


**Artículo 325: The Bible as a Corpus for Language Technologies**


Un corpus para evaluación de QA no demuestra por sí solo consulta documentada con recuperación y generación; no tratar cualquier QA como pertinente.


Abstract:

> question-answering evaluation


### N02


**Artículo 129: Automatic Classification of BI-RADS in Spanish Radiology Reports Using Transformers and Traditional Machine Learning Approaches**


El abstract confirma informes mamográficos y clasificación BI-RADS. El atributo español está en el título, no en el abstract.


Abstract:

> This study aims to classify mammography reports by BI-RADS scores using both transformer-based models and classical machine learning approaches.


Título:

> Automatic Classification of BI-RADS in Spanish Radiology Reports


**Artículo 314: Mammography reporting dataset with BI-RADS system for natural language processing applications: Addressing public data gaps in Spanish**


Recurso textual español etiquetado con BI-RADS; cumple aunque no proponga un clasificador.


Abstract:

> The article presents a Spanish mammography-report dataset with 4,357 records and 15 variables, including complete and sectioned report text, BI-RADS labels and selected metadata.


**Artículo 145: Comparative analysis of generative LLMs for labeling entities in clinical notes**


PLN clínico general. El resumen no enlaza esta contribución con mamografía ni BI-RADS; no asignar parcial por pertenecer a medicina.


Abstract:

> zero-shot labeling of named entities in clinical notes


**Artículo 210: Manual annotation of Robson criteria and obstetric entities: Inter-annotator agreement and initial NER models implementation**


Anotación y NER de registros clínicos, pero de obstetricia y Robson, no mamografía/BI-RADS.


Abstract:

> a manually annotated dataset focused on Robson Criteria Classification


**Artículo 161: ELK: Enhanced Learning Through Cross-Modal Knowledge Transfer for Lesion Detection in Limited-Sample Contrast-Enhanced Mammography Datasets**


Procesamiento de imágenes mamográficas y detección de lesiones; no informes textuales.


Abstract:

> using diffusion models to generate high-resolution, realistic synthetic lesions


**Artículo 372: Added value of feature uncertainty in a radiomic analysis of Contrast-Enhanced Digital Mammography boosted by deep learning**


Radiómica de imágenes y estado inmunohistoquímico; no clasificación del texto por BI-RADS.


Abstract:

> radiomic-based prediction models for contrast-enhanced digital mammographic images


### N03


**Artículo 355: On the Privacy Guarantees of Differentially Private Stochastic Gradient Descent**


Explicita recorte en DP-SGD. También afirma establecer cotas de pérdida de privacidad.


Abstract:

> An inherent feature of this algorithm is the incorporation of gradient clipping to counteract the influence of individual samples during training.


Abstract:

> we establish effective upper bounds for the privacy loss of both projected DP-SGD and regularized DP-SGD


**Artículo 353: Locally Private Sampling with Public Data**


Muestreo con privacidad local. El abstract no dice que analice entrenamiento por gradientes ni recorte; el muestreo privado es una tarea diferente de analizar entrenamiento con gradientes recortados.


Abstract:

> we propose a locally private sampling framework that leverages both the private and public datasets of each user


**Artículo 515: Tensorization of f-Divergences**


Teoría de divergencias con aplicación general a privacidad. No enlaza explícitamente SGD y clipping; no hay base para 1 solo por compartir herramienta matemática.


Abstract:

> it is necessary to evaluate or bound an f-divergence between distributions of random vectors with independent components


**Artículo 185: GIL-IIMAS UNAM at SemEval-2025 Task 4: LA-Min(E): LLM Unlearning Approaches Under Function Minimizing Evaluation Constraints**


Desaprendizaje de LLM; usar gradientes para olvidar información no acredita cotas de privacidad diferencial con recorte.


Abstract:

> Gradient-ascent and task-vector approaches are explored


**Artículo 379: Assessing User Experience with Piezoresistive Force Sensors: Interpreting Button Press Impulse and Duration**


Evaluación de UX con sensores, sin garantía formal del entrenamiento.


Abstract:

> privacy-aware evaluation of user satisfaction in robotic systems


**Artículo 356: Ownership Authentication and Clinical Data Backup in DICOM Images via Robust Watermarking and Binary Data Hiding**


Protección de datos mediante marcas de agua y ocultación, no privacidad del descenso de gradiente.


Abstract:

> two information security levels based on robust watermarking and binary data hiding


### N04


**Artículo 324: text2graphAPI: A library to transform text documents into different graph representations**


Describe biblioteca de conversión. El abstract nombra Information Retrieval y Topic Classification como aplicaciones.


Abstract:

> transforming text documents into different graph representations


Abstract:

> Authorship Analysis, Information Retrieval, and Topic Classification


**Artículo 262: Team iimasnlp at PAN: Leveraging Graph Neural Networks and Large Language Models for Generative AI Authorship Verification**


Conversión explícita y método transferible; objetivo de autoría humano/máquina, no recuperación o temas.


Abstract:

> the first path processes text documents through data augmentation, transforming them into co-occurrence graphs for GNN processing


**Artículo 264: Text Graph Neural Networks for Detecting AI-Generated Content**


Representación texto-grafo; objetivo de detectar generación automática y resumen sin detalle de herramienta.


Abstract:

> combines text-graph representations with graph neural networks and alternative node-feature initialization strategies


**Artículo 266: The iimasNLP team at IberAuTexTification 2024: Integrating Graph Neural Networks, Multilingual LLMs, and Stylometry for Automatic Text Identification**


Clasificación de autoría con GNN, pero el abstract no explica el paso texto→grafo. No inferirlo automáticamente del artículo 262 aunque los trabajos estén relacionados.


Abstract:

> The system architecture incorporates Graph Neural Networks, Multilingual Large Language Models, and stylometric features


**Artículo 150: Contextualized Graph Representations for Generating Counter-Narratives against Hate Speech**


Representaciones conversacionales para contranarrativas. Cercanía de representación; no describe herramienta de documentos→grafos para IR/temas.


Abstract:

> comparing text and graphical representations with varying degrees of context


**Artículo 322: Spectrep: A software for spectral representation of text content**


Software de representación textual, pero el abstract no menciona conversión a grafos. Spectral no equivale automáticamente a graph.


Abstract:

> a multi-view representation that consolidates lexical, syntactic, and semantic features


**Artículo 214: Multi-Label Topic Classification in a Twitter Corpus of Public Communication of Science in Mexican Spanish**


Clasifica temas, pero el abstract enumera ML clásico, transformers y LLM sin grafos; cumplir el destino sin la representación no responde la necesidad.


Abstract:

> automatic multi-label topic classifiers


**Artículo 425: Hypergraph of Functional Connectivity Based on Event-Related Coherence: Magnetoencephalography Data Analysis**


Construcción de grafos de señales cerebrales, no de documentos textuales.


Abstract:

> We construct hypergraphs to analyze functional brain connectivity


### N05


**Artículo 285: A Systematic Literature Review of 10 years of Research on Program Synthesis and Natural Language Processing**


Confirma entrada, salida y problemas de exactitud/robustez y ambigüedad. La pertinencia deriva del abstract, no de ser orientador.


Abstract:

> Program synthesis is the process of automatically generating software from a requirement specification.


Abstract:

> This paper presents a systematic literature review focused on program synthesis from specifications expressed in natural language.


Abstract:

> the complexity of automatically generating accurate and robust code from high-level, ambiguous natural language descriptions


**Artículo 330: Transforming Software Development: A Study on the Integration of Multi-Agent Systems and Large Language Models for Automatic Code Generation**


Aporta evidencia sobre código automático y desafíos; el abstract no confirma que se parta de requisitos expresados en lenguaje natural. El uso de un LLM no permite dar ese detalle por demostrado.


Abstract:

> This paper explores the integration of Multi-Agent Systems (MAS) and Large Language Models (LLMs) for auto-matic code generation


Abstract:

> The research focuses on the technical and theoretical challenges of this integration


**Artículo 327: The Impact of Generative AI Coding Assistants on Developers Who Are Visually Impaired**


Estudia dificultades concretas de asistentes generativos de programación, aunque no confirma síntesis desde requisitos en lenguaje natural. Es una contribución parcial de usabilidad/accesibilidad, no una respuesta completa.


Abstract:

> completed a series of programming tasks using a generative AI coding assistant


Abstract:

> it overwhelmed users with an excessive number of suggestions


**Artículo 465: A fuzzy description logic based IoT framework: Formal verification and end user programming**


Caso fronterizo útil sobre programación de usuario final desde reglas cotidianas y consistencia. No afirmar que produzca código fuente o programas completos si el abstract solo confirma acciones de actuadores.


Abstract:

> lets nontechnical users define smart-environment behavior through everyday fuzzy rules while formally checking rule consistency


Abstract:

> translates vague language into actuator actions


**Artículo 312: LLFSMs to TLA+: A Model-to-Text Transformation of Executable Models Enabling Specification and Verification of Multi-Threaded and Concurrent Systems**


La transformación es entre modelos formales; no responde por sí sola a requisitos en lenguaje natural.


Abstract:

> translating Logic-Labelled Finite-State Machines (LLFSMs) to the Temporal Logic of Actions (TLA)


**Artículo 309: Is It Possible to Use ChatGPT to Perform Measurements Using the COSMIC Method?**


Medir tamaño funcional con un LLM es una tarea distinta de generar programas desde requisitos, aunque el abstract mencione texto y dificultades.


Abstract:

> we investigate how to apply the concepts and guidelines of the COSMIC method to measurements using ChatGPT 4o


### N06


**Artículo 316: Merging Distinct Sources Databases to Improve Software Estimation Models**


El abstract completa la necesidad al proponer una validación para integración de fuentes en estimación de costo/esfuerzo. Es un ejemplo directo sin convertirlo en la única respuesta posible.


Abstract:

> the arbitrary combination of different source databases belonging to practitioners in order to create larger datasets


Abstract:

> Propose the application of the Kruskal–Wallis test to validate the integration of distinct source databases (independent groups), thereby avoiding the mixing of unrelated data, increasing the number of data points, and improving the estimation models.


**Artículo 309: Is It Possible to Use ChatGPT to Perform Measurements Using the COSMIC Method?**


El contexto de estimación de software es cercano, pero el aporte trata de medición funcional con ChatGPT, no de adecuación de combinar fuentes. No asignar 1 únicamente por compartir el contexto.


Abstract:

> we investigate how to apply the concepts and guidelines of the COSMIC method to measurements using ChatGPT 4o


**Artículo 320: Q-COSMIC: Quantum Software Metrics Based on COSMIC (ISO/IEC19761)**


Una técnica de tamaño funcional no demuestra criterios de integración de datos de proyectos para estimación de esfuerzo.


Abstract:

> Here we introduce Q-COSMIC, a technique for measuring the functional size of quantum software


**Artículo 213: MMDA: A Multimodal and Multisource Domain Adaptation Method for Cross-Subject Emotion Recognition From EEG and Eye Movement Signals**


Aunque estudia alineación entre fuentes y transferencia negativa, el objetivo es reconocimiento de emociones con EEG/ojos. La cercanía metodológica no responde a la necesidad específica de estimación de software.


Abstract:

> Multimodal emotion recognition from electroencephalogram (EEG) and eye movement signals


Abstract:

> relevant distribution alignments must be prioritized to avoid a negative transfer


### N07


**Artículo 336: Web-based wide-area monitoring platform for ringdown and clustering analytics in power systems**


Confirma todos los componentes centrales; las tecnologías concretas y el país son descriptivos, no requisitos de pertinencia.


Abstract:

> an open-source research platform for monitoring the Mexican interconnected power grid, allowing real-time processing and information extraction of the grid's dynamic condition


Abstract:

> embeds different ringdown and clustering analytics tools


Abstract:

> the coherent behaviour of generator and non-generator buses is provided


**Artículo 444: Power System Coherency via Taylor-Fourier Modal Extraction and Elbow-Based Clustering on Merging Costs**


El método conecta mediciones, modos y coherencia; es altamente útil para un componente, pero el abstract no presenta una plataforma integrada como pide la consulta.


Abstract:

> addresses power system coherency by extracting oscillatory modal characteristics


Abstract:

> The phase of the slower frequency modes is extracted from rotor speed measurements after a large disturbance occurs


**Artículo 383: Bessel and cosine filtering approaches for electromechanical modes identification**


Proporciona identificación modal de señales PMU; no confirma agrupación por coherencia ni una plataforma con ambas funciones.


Abstract:

> identify modal patterns (damping and frequency) from multivariate and multi-modal oscillation signals stemming from multiple PMUs and locations


**Artículo 30: Control of Electromechanical Oscillations with Machine Learning**


Relaciona clustering con oscilaciones electromecánicas, útil como método; el fin principal es control y no documenta una plataforma de análisis integrada.


Abstract:

> uses the HACA clustering method to design electromechanical oscillation control


Abstract:

> The system is identified by the Eigensystem Realization Algorithm (ERA) for clustering and for tuning the LQG control


**Artículo 70: Power System Stabilizers’ Design Through The Eigensystem Realization Algorithm**


Aporta identificación modal desde mediciones como componente; no documenta clustering de coherencia ni una plataforma integral.


Abstract:

> derives a state-space representation using power system measurements to compute the power system eigenvalues and their residues


**Artículo 446: PQ-SyDa: power quality synthetic disturbances dataset**


Que sea una GUI eléctrica no basta: genera eventos sintéticos de calidad de energía y no demuestra el análisis combinado de oscilaciones/coherencia.


Abstract:

> a Phyton-based user-friendly graphical user interface (GUI) toolbox to synthetically generate power quality events dataset


**Artículo 178: Framework for training and deployment machine learning methods in real-time simulator: short-term kinetic energy forecasting in power systems**


Es una plataforma de pronóstico de energía cinética/inercia; el abstract no confirma identificación modal ni grupos coherentes. Evitar darle pertinencia solo por platform + power system.


Abstract:

> a framework for training and deploying machine learning methods for real-time power systems' kinetic energy forecasting


### N08


**Artículo 1: A comparative study of NeuralODE and Universal ODE approaches to solving Chandrasekhar White Dwarf equation**


El abstract apoya método, problema físico y fallos del pronóstico. Preservar su terminología, sin inferir que se predice evolución temporal de una estrella real.


Abstract:

> Neural Ordinary Differential Equations (Neural ODEs) and Universal Differential Equations (UDEs) to the Chandrasekhar White Dwarf Equation (CWDE)


Abstract:

> describes the relationship between the density of the white dwarf and its distance from the center


Abstract:

> both Neural ODEs and UDEs can be used effectively for both prediction as well as forecasting of the CWDE


Abstract:

> we introduce the 'forecasting breakdown point' - the time at which forecasting fails for both Neural ODEs and UDEs


**Artículo 67: Physics-Informed Neural Networks for Higher-Order Nonlinear Schrödinger Equations: Soliton Dynamics in External Potentials**


Usa redes para ecuaciones diferenciales pero trata solitones/NLSE, no estructura de enanas blancas; la técnica amplia compartida no satisface esta necesidad específica.


Abstract:

> physics-informed neural networks (PINNs) for solving higher-order nonlinear partial differential equations belonging to the nonlinear Schrödinger equation (NLSE) hierarchy


**Artículo 99: Verified propagation of imprecise probabilities in non-linear ODEs**


La presencia de ODE, predicción y failure en este documento no implica redes neuronales ni enanas blancas.


Abstract:

> We combine reachability analysis and probability bounds analysis


Abstract:

> failure probability calculations in reliability analysis


**Artículo 91: tonalli: an asexual genetic code to characterize APOGEE-2 stellar spectra. I. Validation with synthetic and solar spectra**


Caracteriza espectros estelares mediante algoritmo genético. Astronomía y limitaciones por sí solas no responden al modelo físico y método pedidos.


Abstract:

> tonalli implements an asexual genetic algorithm


Abstract:

> we study the performance, limitations, accuracy, and precision of our tool


### N09


**Artículo 4: A Hierarchical Approach to a Tri-Objective Portfolio Optimization Problem Considering an ESG Index**


El resumen conecta explícitamente portafolio, ESG, retorno y riesgo; además describe un procedimiento de optimización.


Abstract:

> This addition empowers investors to customize their portfolios by defining explicit trade-off thresholds between expected returns and risk, considering the ESG index.


**Artículo 97: Using exploratory modeling to challenge narratives of risk governance in Mexico City**


Riesgo, sostenibilidad y gobernanza se refieren al agua y al territorio, no al compromiso financiero de un portafolio.


Abstract:

> a dynamic, exploratory model of socio-hydrological risks in Mexico City


**Artículo 260: Sustainable visions: unsupervised machine learning insights on global development goals**


Los indicadores de desarrollo sostenible de países no equivalen a integrar ESG en carteras financieras.


Abstract:

> The study applies unsupervised machine-learning methods to patterns among global sustainable-development indicators


**Artículo 487: Invited Paper: Further Decentralizing Decentralized Finance: Loss vs Rebalancing Without a Shared Reference Market**


Está en finanzas y rebalanceo, pero el resumen no aporta el criterio ESG ni su integración en riesgo-rendimiento de un portafolio.


Abstract:

> Automated market makers (AMMs) typically rely on arbitrage agents to keep prices in line with a shared reference market


**Artículo 497: On the Approximation of the Entire Pareto Front of a Constrained Multi-objective Optimization Problem**


La optimización multiobjetivo general no se vuelve pertinente solo por compartir técnicas con el artículo orientador; no describe una aplicación de portafolio o ESG.


Abstract:

> we consider the problem of capturing the entire Pareto front


### N10


**Artículo 271: TopoGCL: Topological Graph Contrastive Learning**


El resumen identifica la omisión de subestructuras de orden superior y explica cómo incorpora topología a GCL.


Abstract:

> We address this limitation by introducing the concepts of topological invariance and extended persistence on graphs to GCL.


**Artículo 386: Brain Connectivity Hypergraphs**


Representar conectividad cerebral con hipergrafos no demuestra un método de aprendizaje contrastivo ni una integración equivalente.


Abstract:

> we defined a functional network connecting different brain regions (vertices) across time segments by generating edge time series that capture temporal fluctuations in connectivity.


**Artículo 424: Hypergraph Analysis of Functional Brain Connectivity During Figurative Attention**


La presencia de relaciones de orden superior no satisface por sí misma el requisito de aprendizaje contrastivo.


Abstract:

> including previously unobserved higher-order relationships between brain regions


**Artículo 457: Topological Data Analysis in Cardiovascular Signals: An Overview**


Una revisión de TDA en señales cardiovasculares puede servir como bibliografía general, pero el resumen no conecta esas técnicas con el problema contrastivo de grafos solicitado.


Abstract:

> We first introduce the concept of TDA and its key techniques, including persistent homology, Mapper, and multidimensional scaling.


**Artículo 54: Modeling Higher-Order Interactions in Sparse and Heavy-Tailed Neural Population Activity**


Las interacciones de neuronas y distribuciones de actividad no equivalen a subestructuras de grafos usadas en GCL.


Abstract:

> joint activity of homogeneous binary neurons generates sparse and widespread population firing rate distributions


**Artículo 262: Team iimasnlp at PAN: Leveraging Graph Neural Networks and Large Language Models for Generative AI Authorship Verification**


Que se utilicen GNN y grafos de coocurrencia no indica topología de orden superior ni aprendizaje contrastivo.


Abstract:

> This study presents a novel model architecture integrating Graph Neural Networks (GNNs), pre-trained Language Models (LLMs), and stylometric features to classify text documents as human-generated or machine-generated.


### N11


**Artículo 337: A Parallel Steganography Algorithm Using a New Secret Key Generation**


El resumen acredita los tres elementos y añade que el paralelismo mejora ambas fases de codificación/decodificación.


Abstract:

> The research work in this paper explains how secret key steganography, combined with parallel processing, is utilized to conceal secret data within an image.


**Artículo 350: Enhancing the Capacity and Robustness of an LSB Algorithm Using a Novel Insertion Method, Hashing Function, and Secret Key**


Hay esteganografía de imágenes y clave secreta; el resumen menciona velocidad y tiempo, pero no atribuye la aceleración a procesamiento paralelo.


Abstract:

> The present paper explains how secret key steganography uses a new insertion method to hide information inside an image and a hashing function applied on the secret data.


**Artículo 342: Color Image Protection via Quantum Encryption and HVS-Based Mini-Block Data Hiding**


Hay inserción de información y cifrado combinado. La frase posterior «In parallel, the encryption stage...» no basta para acreditar ejecución paralela computacional; tampoco se documenta en el resumen la clave secreta de la inserción.


Abstract:

> HVS-based encoding tables are used to embed the information within these blocks


**Artículo 356: Ownership Authentication and Clinical Data Backup in DICOM Images via Robust Watermarking and Binary Data Hiding**


La marca de agua sí inserta datos en una imagen y aporta un mecanismo de ocultamiento, pero el resumen no establece clave secreta ni procesamiento paralelo.


Abstract:

> Firstly, clinical data is embedded into a logo using binary data hiding. Then, this logo is inserted into the spatial domain of a medical image


**Artículo 352: Improved protection for colour images via invertible colour-to-grey, image distortion, and visible-imperceptible watermarking**


El ocultamiento de información cromática es un mecanismo de inserción; recuperar más rápido no prueba paralelización. La clave secreta tampoco queda acreditada en el resumen.


Abstract:

> These schemes hide the colour information into a grayscale version of the image


**Artículo 362: Symmetric Cipher for Grayscale Images Based on Quantum Differential Geometry**


El artículo usa claves y protege imágenes mediante cifrado, pero no describe inserción de un mensaje en una imagen portadora; no satisface el núcleo esteganográfico.


Abstract:

> In the first phase, a subkey is generated through an addition and multiplication process in Z_16 based on a user-defined key.


**Artículo 359: Robust Image Tampering Detection and Ownership Authentication Using Zero-Watermarking and Siamese Neural Networks**


El vínculo entre características y marcas no acredita inserción de información en una imagen portadora. El resumen tampoco acredita los requisitos de clave y paralelismo.


Abstract:

> A feature matrix is generated from the owner's portrait using the Siamese network and securely linked to the image's halftone watermark through an XOR operation.


**Artículo 338: A Protocol for Solving Certificate Poisoning for the OpenPGP Keyserver Network**


La distribución de claves OpenPGP y el envenenamiento de certificados no aportan un algoritmo para ocultar información en imágenes.


Abstract:

> using a non-authenticated, distributed keyserver network for key distribution and discovery


### N12


**Artículo 482: Cops and Robbers on Token Graphs**


El resumen responde exactamente a las familias y a la magnitud solicitadas. El resultado sobre estrellas subdivididas es adicional, no redefine la consulta.


Abstract:

> We obtain the exact cop number for k-token graphs of paths and stars.


**Artículo 501: Ortho-Unit Polygons can be Guarded with at most floor((n-4)/8) Guards**


Contar guardias para polígonos de unidades no equivale al número de policías de un juego en grafos de fichas.


Abstract:

> every such polygon with at least twelve vertices can be guarded using at most floor((n-4)/8) guards


**Artículo 506: Realizable signatures in upward pointset embeddings of directed paths**


Los caminos aparecen en incrustaciones geométricas; no hay juego de persecución ni construcción de grafos de fichas.


Abstract:

> we consider non-self-intersecting paths over sets of n + 1 points in general position with distinct y-coordinates


**Artículo 339: A State of the Art of Non-fungible Tokens: A Literature Review**


Los tokens financieros no son fichas/vertices de la construcción matemática de token graphs.


Abstract:

> Non-fungible tokens, also known as NFTs, are based on the blockchain


**Artículo 490: Maya-Tupi graphs: A generalization of split graphs**


Se estudian clases y caracterizaciones de grafos, no el juego de policías y ladrones en grafos de fichas de caminos o estrellas.


Abstract:

> We find characterizations in terms of minimal forbidden induced subgraphs for disconnected graphs, trees and cographs